# START HERE — the minimal tabular_agent product

One table, one target, one train/evaluation split, one fixed model, one replayable bundle. Cross-fitting exists only inside fitted encoders; the runtime exposes no CV score, bootstrap error, block spread, sealed score or model blend.

In [ ]:
import os
import tempfile

import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 400
df = pd.DataFrame({
    'area': rng.uniform(30, 180, n),
    'city': rng.choice(['berlin', 'hamburg', 'koeln'], n),
})
df['price'] = 900 * df['area'] + df['city'].map({'berlin': 20000, 'hamburg': 8000, 'koeln': 0}) + rng.normal(0, 5000, n) + 30000
workdir = tempfile.mkdtemp(prefix='tabular_agent_start_')
data_path = os.path.join(workdir, 'houses.csv')
df.to_csv(data_path, index=False)
df.head()

## 1. Inspect, then build immutable candidate tables

The evaluation rows are split away before any tool runs. Strings are unusable until a fitted encoder adds a replayable numeric column.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import fit_encoding, inspect_data
from tabular_agent.taskspec import TaskSpec

session = RunSession(
    os.path.join(workdir, 'run'), data_path, TaskSpec(target='price'),
    profile='minimal',
)
info = inspect_data(session)
encoded = fit_encoding(session, ['city'], method='frequency')
print('metric:', info['metric'])
print('evaluation score:', encoded['evaluation_score'])
print('gain vs parent:', encoded['gain_vs_parent'])

## 2. Finalize once

The best candidate — including the data as given — receives one fixed LGBM fit. No terminal cross-validation or model-family blend is performed.

In [ ]:
from tabular_agent.harness.tools import finalize_run

result = finalize_run(session)
{key: result[key] for key in ['model_type', 'metric', 'evaluation_score', 'baseline_score', 'bundle_dir']}

## 3. Replay from raw rows

The bundle carries the fitted encoder and model. Prediction starts from the original columns; no target and no precomputed feature table are required.

In [ ]:
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(session.run_dir, result['bundle_dir']))
raw = df.drop(columns=['price']).head(8)
predictions = predict_bundle(bundle, raw)
assert len(predictions) == 8 and np.isfinite(predictions).all()
predictions

## Research is separate

Historical CV comparisons, uncertainty diagnostics, forecasting streams, model blends, competition runners and negative probes are preserved under `research/archive/`. They are evidence about the product, not concepts every product run must expose.